In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
ROLE_PROFILE_PATH = "skill_gap/role_skill_profiles.csv"

role_profiles = pd.read_csv(ROLE_PROFILE_PATH)
role_profiles.head()


FileNotFoundError: [Errno 2] No such file or directory: 'career-ml/skill_gap/role_skill_profiles.csv'

In [1]:
role_skill_matrix = role_profiles.pivot_table(
    index="role_id",
    columns="skill_id",
    values="importance",
    fill_value=0
)

role_skill_matrix.shape


NameError: name 'role_profiles' is not defined

In [ ]:
def build_user_vector(user_skill_ids, skill_columns):
    """
    Convert a user's skill IDs into a vector
    aligned with role_skill_matrix columns.
    """
    vector = np.zeros(len(skill_columns))
    skill_index = {skill: idx for idx, skill in enumerate(skill_columns)}

    for skill_id in user_skill_ids:
        if skill_id in skill_index:
            vector[skill_index[skill_id]] = 1.0

    return vector.reshape(1, -1)


In [ ]:
def recommend_careers(user_skill_ids, top_n=5):
    """
    Recommend best-fit career roles based on cosine similarity.
    """
    user_vector = build_user_vector(
        user_skill_ids,
        role_skill_matrix.columns
    )

    role_vectors = role_skill_matrix.values
    similarity_scores = cosine_similarity(user_vector, role_vectors)[0]

    recommendations = pd.DataFrame({
        "role_id": role_skill_matrix.index,
        "match_score": similarity_scores
    }).sort_values("match_score", ascending=False)

    return recommendations.head(top_n)


In [ ]:
user_skill_ids = {
    "SK004",  # python
    "SK031",  # machine learning
    "SK102"   # sql (example)
}

recommend_careers(user_skill_ids, top_n=5)


In [ ]:
def get_best_career(user_skill_ids):
    top = recommend_careers(user_skill_ids, top_n=1)
    return top.iloc[0]["role_id"], top.iloc[0]["match_score"]
